In [2]:
!pip install fastapi uvicorn pyngrok nest-asyncio

In [3]:
from huggingface_hub import notebook_login

notebook_login()


In [4]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 40.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 51.7 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [5]:

# Load model directly
from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained("google/gemma-4-12B-it")
model = AutoModelForMultimodalLM.from_pretrained("google/gemma-4-12B-it",
                                                     max_memory={
                                                        0: "13GiB",
                                                        1: "13GiB",
                                                        "cpu": "30GiB"
                                                    },device_map="auto")

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/23.9G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

In [6]:
import asyncio, json, re, logging, time
import math
import torch

# ============================================================
# UTILIDADES DE ANÁLISIS
# ============================================================

def token_es_nota(token):
    """
    Determina si el token representa exactamente una nota entre 0 y 10.
    """
    token_limpio = token.strip()

    return (
        token_limpio.isdigit()
        and 0 <= int(token_limpio) <= 10
        and token_limpio == str(int(token_limpio))
    )


def analizar_distribucion(logits, tokenizer, top_x_candidatos=3):
    """
    Analiza la distribución de probabilidad tomando los Top X candidatos 
    de mayor probabilidad del vocabulario (por defecto Top 3).
    """
    # --------------------------------------------------------
    # PROBABILIDADES
    # --------------------------------------------------------
    probs = torch.softmax(logits, dim=-1)

    # --------------------------------------------------------
    # TOP X CANDIDATOS DEL VOCABULARIO COMPLETO
    # --------------------------------------------------------
    probs_cpu = probs.detach().cpu()
    
    probs_ordenadas, indices_ordenados = torch.sort(
        probs_cpu,
        descending=True
    )

    # Tomamos los Top X tokens más probables
    top_x_probs = probs_ordenadas[:top_x_candidatos]
    top_x_indices = indices_ordenados[:top_x_candidatos]

    candidatos_vocabulario = []
    for prob, idx in zip(top_x_probs, top_x_indices):
        token = tokenizer.decode([idx.item()])
        candidatos_vocabulario.append({
            "token": token,
            "token_id": idx.item(),
            "probabilidad": prob.item(),
            "es_nota": token_es_nota(token)
        })

    # Métrica: Masa de probabilidad acumulada de los Top X tokens
    masa_acumulada_top_x = top_x_probs.sum().item()

    # --------------------------------------------------------
    # NOTAS 0-10
    # --------------------------------------------------------
    notas = []

    for nota in range(11):
        token_ids = tokenizer.encode(
            str(nota),
            add_special_tokens=False
        )

        if len(token_ids) != 1:
            continue

        token_id = token_ids[0]
        prob = probs[token_id].item()
        logit = logits[token_id].item()

        notas.append({
            "nota": nota,
            "token": tokenizer.decode([token_id]),
            "token_id": token_id,
            "probabilidad": prob,
            "logit": logit
        })

    notas.sort(
        key=lambda x: x["probabilidad"],
        reverse=True
    )

    # --------------------------------------------------------
    # MASAS DE NOTAS Y TOP X NOTAS
    # --------------------------------------------------------
    masa_notas = sum(x["probabilidad"] for x in notas)
    masa_no_notas = 1.0 - masa_notas

    candidatos_notas_top_x = notas[:top_x_candidatos]
    masa_notas_top_x_acumulada = sum(x["probabilidad"] for x in candidatos_notas_top_x)

    # --------------------------------------------------------
    # NOTA PREDICHA Y MARGENES
    # --------------------------------------------------------
    nota_predicha = notas[0]["nota"] if notas else None
    probabilidad_predicha = notas[0]["probabilidad"] if notas else None

    if len(notas) >= 2:
        margen_logit = notas[0]["logit"] - notas[1]["logit"]
        diferencia_probabilidad = notas[0]["probabilidad"] - notas[1]["probabilidad"]
    else:
        margen_logit = None
        diferencia_probabilidad = None

    # --------------------------------------------------------
    # ENTROPÍA Y PERPLEJIDAD
    # --------------------------------------------------------
    entropia = -sum(
        p.item() * math.log2(p.item())
        for p in probs
        if p.item() > 0
    )

    perplejidad = 2 ** entropia

    # --------------------------------------------------------
    # RESULTADO
    # --------------------------------------------------------
    return {
        "nota_predicha": nota_predicha,
        "probabilidad_predicha": probabilidad_predicha,

        # Métricas de Top X general
        "top_x_solicitados": top_x_candidatos,
        "candidatos_vocabulario": candidatos_vocabulario,
        "masa_acumulada_top_x": masa_acumulada_top_x,

        # Métricas relativas a las notas
        "notas": notas,
        "candidatos_notas_top_x": candidatos_notas_top_x,
        "masa_notas": masa_notas,
        "masa_no_notas": masa_no_notas,
        "masa_notas_top_x_acumulada": masa_notas_top_x_acumulada,

        # Márgenes
        "margen_logit": margen_logit,
        "diferencia_probabilidad": diferencia_probabilidad,

        # Métricas de incertidumbre
        "entropia": entropia,
        "perplejidad": perplejidad
    }


# ============================================================
# COMPONENTE LLM
# ============================================================

class LLM:

    def complete(self, messages):
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            add_generation_prompt=True,
            enable_thinking=False
        ).to(model.device)
        
        input_len = inputs["input_ids"].shape[-1]
        
        # Generar activando la captura de logits (scores)
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=40,
                return_dict_in_generate=True,
                output_scores=True
            )
        
        # 1. Analizar la distribución usando los logits del PRIMER token generado
        logits_primer_token = outputs.scores[0][0]  # Dimensión [vocab_size]
        metricas = analizar_distribucion(
            logits=logits_primer_token, 
            tokenizer=processor.tokenizer, 
            top_x_candidatos=3
        )

        # 2. Decodificar y limpiar la respuesta del modelo
        response = processor.decode(outputs.sequences[0][input_len:], skip_special_tokens=True)
        response = re.sub(r'\|', '', response)
        response = re.split(r'<turn>', response)[0].strip()

        # Retorna el texto generado junto a las métricas del Top 3
        return {
            "response": response,
            "metricas": metricas
        }

In [7]:
llm = LLM()

from pydantic import BaseModel
from typing import List, Optional, Literal
from datetime import datetime

class Message(BaseModel):
    role: Literal['system', 'user']  # roles comunes en chats
    content: str

class ChatRequest(BaseModel):
    messages: List[Message] = []   # Valor por defecto lista vacía

In [14]:
from fastapi import FastAPI
from pyngrok import ngrok
import nest_asyncio
import uvicorn

ngrok.set_auth_token("3HVPxNKP91g3wQqtOmFHLJkXYTI_6aVvVGh2J5VQkkjYPYjn5")

import math
import numpy as np  # Si usas numpy en Colab

from fastapi.encoders import jsonable_encoder
from fastapi.responses import JSONResponse

def sanitizar_metricas(data):
    """
    Convierte valores inf, -inf, nan a None o strings para que sea JSON estándar válido,
    y desenvuelve tipos de numpy o tensores de PyTorch.
    """
    # Si es un tensor de PyTorch con un solo valor
    if isinstance(data, torch.Tensor):
        data = data.item() if data.numel() == 1 else data.tolist()

    # Si es un tipo escalar de numpy (np.float32, np.float64, etc.)
    if isinstance(data, np.generic):
        data = data.item()

    if isinstance(data, dict):
        return {k: sanitizar_metricas(v) for k, v in data.items()}
    elif isinstance(data, (list, tuple)):
        return [sanitizar_metricas(v) for v in data]
    elif isinstance(data, (float, int)):
        if math.isinf(data) or math.isnan(data):
            return None  # En JSON se convierte en null (o puedes poner "-inf" / 0.0)
        return data
    elif hasattr(data, "model_dump"):  # Si es un modelo Pydantic
        return sanitizar_metricas(data.model_dump())
    elif hasattr(data, "__dict__"):
        return sanitizar_metricas(vars(data))

    return data


app = FastAPI()

@app.get("/")
def home():
    return {"message": "Hola desde Colab + FastAPI!"}


@app.post("/chat")
async def chat(body: ChatRequest):
    messages = [m.model_dump() for m in body.messages]
    response = llm.complete(messages)
    
    # Sanitizamos para eliminar -inf/inf/nan y convertir tensores
    sanitized = sanitizar_metricas(response)
    
    # Si response ya es un dict con {"response": ..., "metricas": ...}, lo retornamos directamente
    if isinstance(sanitized, dict):
        return JSONResponse(content=jsonable_encoder(sanitized))
    
    # Si response solo era un string con el texto generado
    return JSONResponse(content={"response": sanitized, "metricas": {}})


In [ ]:
# Crear túnel en el puerto 8000
public_url = ngrok.connect(8000)
print("URL pública:", public_url)

config = uvicorn.Config(app=app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()

URL pública: NgrokTunnel: "https://jolt-getup-cylinder.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2803:9800:9885:a4dc:35ad:c3f:fc7e:15ee:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:35ad:c3f:fc7e:15ee:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:35ad:c3f:fc7e:15ee:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:35ad:c3f:fc7e:15ee:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:35ad:c3f:fc7e:15ee:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:35ad:c3f:fc7e:15ee:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:35ad:c3f:fc7e:15ee:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:35ad:c3f:fc7e:15ee:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:35ad:c3f:fc7e:15ee:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:35ad:c3f:fc7e:15ee:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:35ad:c3f:fc7e:15ee:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:35ad:c3f:fc7e:15ee:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9

In [9]:
print(model.hf_device_map)

{'model.language_model.embed_tokens': 0, 'lm_head': 0, 'model.language_model.layers.0': 0, 'model.language_model.layers.1': 0, 'model.language_model.layers.2': 0, 'model.language_model.layers.3': 0, 'model.language_model.layers.4': 0, 'model.language_model.layers.5': 0, 'model.language_model.layers.6': 0, 'model.language_model.layers.7': 0, 'model.language_model.layers.8': 0, 'model.language_model.layers.9': 0, 'model.language_model.layers.10': 0, 'model.language_model.layers.11': 0, 'model.language_model.layers.12': 0, 'model.language_model.layers.13': 0, 'model.language_model.layers.14': 0, 'model.language_model.layers.15': 0, 'model.language_model.layers.16': 0, 'model.language_model.layers.17': 0, 'model.language_model.layers.18': 0, 'model.language_model.layers.19': 0, 'model.language_model.layers.20': 0, 'model.language_model.layers.21': 0, 'model.language_model.layers.22': 1, 'model.language_model.layers.23': 1, 'model.language_model.layers.24': 1, 'model.language_model.layers.2